In [ ]:
# Load env variables and create client
from pathlib import Path
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
# model = "claude-haiku-4-5"
# base_url = "http://127.0.0.1:8045"
# api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
base_url = os.getenv("BASE_URL")
api_key = os.getenv("API_KEY")
claude_code_headers = {
           "User-Agent": "claude-code/1.0.24",
           "X-Stainless-Lang": "js",
           "X-Stainless-Package-Version": "0.52.0",
           "X-Stainless-OS": "MacOS",
           "X-Stainless-Arch": "arm64",
           "X-Stainless-Runtime": "node",
           "X-Stainless-Runtime-Version": "v22.13.1",
           "X-Stainless-Async": "async:promise",
       }

client = Anthropic(api_key=api_key, base_url=base_url, default_headers=claude_code_headers)
# model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"
model = "claude-sonnet-4-5"

In [14]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [15]:
file_metadata = upload("./feature_of_claude/streaming.csv")
file_metadata

AuthenticationError: Error code: 401 - {'error': {'code': '', 'message': '未提供令牌 (request id: 2026022416552872275238JCnf98fB)', 'type': 'new_api_error'}}

In [ ]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
进行详细分析以确定客户流失的主要驱动因素。你的最终输出应至少包含一张总结分析结果的详细图表。

每次执行代码时，都是从一个全新的空白环境开始。之前执行中的变量和库导入都不存在，你需要重新声明/重新导入所有变量和库。
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

In [ ]:
download_file("file_011CPYZqxoMSsfbrSzFw8j9X")